In [ ]:
# V3 VVA training: intent-relative visual prompt + delta actions + separate arm/gripper heads.
# Put this notebook in Colab, select a T4 GPU runtime, and run this single cell.
# It expects dataset_2d.npy and saves/downloads model3_delta_intent.pth.

import math
import random
from pathlib import Path

import numpy as np
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader, Dataset, Subset

try:
    from google.colab import files
    IN_COLAB = True
except Exception:
    files = None
    IN_COLAB = False


SEED = 42
DATA_PATH = Path('dataset_2d.npy')
MODEL_OUT = Path('model3_delta_intent.pth')

RAW_POINT_DIM = 14
INTENT_FEATURE_DIM = 24
INPUT_DIM = INTENT_FEATURE_DIM
ACTION_DIM = 6
H_WINDOW = 10
PHASE_DIM = 5

EMBED_DIM = 192
NHEAD = 4
NUM_LAYERS = 4
DROPOUT = 0.10

BATCH_SIZE = 64
EPOCHS = 750
LR = 3e-4
WEIGHT_DECAY = 1e-4
CAM_NOISE_STD = 0.012
DEMO_NOISE_STD = 0.018
PROGRESS_NOISE_STD = 0.006
GRIPPER_LOSS_WEIGHT = 3.0

random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
torch.cuda.manual_seed_all(SEED)

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print('Device:', device)


if not DATA_PATH.exists():
    if not IN_COLAB:
        raise FileNotFoundError('dataset_2d.npy was not found in the current folder.')
    print('Upload dataset_2d.npy now.')
    uploaded = files.upload()
    if not uploaded:
        raise RuntimeError('No file was uploaded.')
    DATA_PATH = Path(next(iter(uploaded.keys())))

raw_data = np.load(DATA_PATH, allow_pickle=True)
print(f'Loaded {len(raw_data)} trials from {DATA_PATH}')


def as_float_array(x):
    return np.asarray(x, dtype=np.float32)


def points14_to_intent_features(points14, eps=1e-6):
    arr = np.asarray(points14, dtype=np.float32)
    if arr.shape[-1] != RAW_POINT_DIM:
        raise ValueError(f'Expected last dimension {RAW_POINT_DIM}, got {arr.shape[-1]}')

    leading_shape = arr.shape[:-1]
    pts = arr.reshape(-1, 7, 2)

    base = pts[:, 0]
    shoulder = pts[:, 1]
    elbow = pts[:, 2]
    wrist = pts[:, 3]
    grip_l = pts[:, 4]
    grip_r = pts[:, 5]
    target = pts[:, 6]

    ee = 0.5 * (grip_l + grip_r)
    scale = (
        np.linalg.norm(shoulder - base, axis=1)
        + np.linalg.norm(elbow - shoulder, axis=1)
        + np.linalg.norm(wrist - elbow, axis=1)
        + np.linalg.norm(ee - wrist, axis=1)
    )
    scale = np.maximum(scale, eps).reshape(-1, 1)

    centered_points = ((pts - base[:, None, :]) / scale[:, None, :]).reshape(-1, 14)
    ee_rel_base = (ee - base) / scale
    target_rel_ee = (target - ee) / scale
    grip_vec = (grip_r - grip_l) / scale

    grip_width = np.linalg.norm(grip_r - grip_l, axis=1, keepdims=True) / scale
    target_dist = np.linalg.norm(target - ee, axis=1, keepdims=True) / scale
    target_dir = target_rel_ee / np.maximum(target_dist, eps)

    features = np.concatenate(
        [
            centered_points,
            ee_rel_base,
            target_rel_ee,
            grip_vec,
            grip_width,
            target_dist,
            target_dir,
        ],
        axis=1,
    ).astype(np.float32)
    return features.reshape(*leading_shape, INTENT_FEATURE_DIM)


def sequence_to_intent_features(sequence):
    sequence = np.asarray(sequence, dtype=np.float32)
    if sequence.size == 0:
        return np.empty((0, INTENT_FEATURE_DIM), dtype=np.float32)
    return points14_to_intent_features(sequence)


def phase_features(progress):
    p = progress.clamp(0.0, 1.0)
    return torch.cat(
        [
            p,
            torch.sin(math.pi * p),
            torch.cos(math.pi * p),
            torch.sin(2.0 * math.pi * p),
            torch.cos(2.0 * math.pi * p),
        ],
        dim=1,
    )


class DeltaIntentDataset(Dataset):
    def __init__(self, trials, h_window=10):
        self.h_window = h_window
        self.samples = []
        self.trial_numbers = []

        converted_trials = []
        all_deltas = []

        for fallback_trial_id, trial in enumerate(trials, start=1):
            trial_number = int(trial.get('trial', fallback_trial_id))
            demo = sequence_to_intent_features(as_float_array(trial['demo_X']))
            cam = sequence_to_intent_features(as_float_array(trial['cam_X']))
            actions = as_float_array(trial['actions'])
            n = min(len(cam), len(actions))
            if len(demo) == 0 or n < 2:
                continue

            deltas = actions[1:n] - actions[:n - 1]
            all_deltas.append(deltas)
            converted_trials.append((trial_number, demo, cam[:n], actions[:n], deltas))

        if not converted_trials:
            raise RuntimeError('No usable trials found. Need at least two action frames per trial.')

        self.max_demo_len = max(len(demo) for _, demo, _, _, _ in converted_trials)
        all_deltas = np.concatenate(all_deltas, axis=0)
        self.delta_mean = all_deltas.mean(axis=0).astype(np.float32)
        self.delta_std = np.maximum(all_deltas.std(axis=0), 1e-5).astype(np.float32)

        for trial_number, demo, cam, actions, deltas in converted_trials:
            pad_len = self.max_demo_len - len(demo)
            demo_padded = np.pad(demo, ((0, pad_len), (0, 0)), mode='constant')
            demo_mask = np.zeros(self.max_demo_len, dtype=bool)
            demo_mask[len(demo):] = True

            # Input is visual state up to time t. Target is the next delta action t -> t+1.
            for t in range(len(deltas)):
                if t < h_window:
                    prefix = np.tile(cam[0], (h_window - t - 1, 1))
                    hist = np.vstack([prefix, cam[:t + 1]])
                else:
                    hist = cam[t - h_window + 1:t + 1]

                progress = 0.0 if len(deltas) <= 1 else t / float(len(deltas) - 1)
                target_delta = (deltas[t] - self.delta_mean) / self.delta_std

                self.samples.append({
                    'demo_seq': torch.tensor(demo_padded, dtype=torch.float32),
                    'demo_mask': torch.tensor(demo_mask, dtype=torch.bool),
                    'cam_hist': torch.tensor(hist.flatten(), dtype=torch.float32),
                    'progress': torch.tensor([progress], dtype=torch.float32),
                    'delta': torch.tensor(target_delta, dtype=torch.float32),
                    'trial_number': trial_number,
                })
                self.trial_numbers.append(trial_number)

        print('Feature mode: intent_relative')
        print('Action mode: delta')
        print('Feature dim:', INTENT_FEATURE_DIM)
        print('Max demo length:', self.max_demo_len)
        print('Samples:', len(self.samples))
        print('Trials:', sorted(set(self.trial_numbers)))
        print('Delta mean:', self.delta_mean)
        print('Delta std:', self.delta_std)

    def __len__(self):
        return len(self.samples)

    def __getitem__(self, idx):
        return self.samples[idx]


class DeltaIntentPolicy(nn.Module):
    def __init__(
        self,
        input_dim=24,
        h_window=10,
        phase_dim=5,
        embed_dim=192,
        action_dim=6,
        nhead=4,
        num_layers=4,
        dropout=0.10,
        max_tokens=2000,
    ):
        super().__init__()
        self.max_tokens = max_tokens

        self.demo_embed = nn.Sequential(
            nn.Linear(input_dim, embed_dim),
            nn.LayerNorm(embed_dim),
            nn.GELU(),
            nn.Linear(embed_dim, embed_dim),
        )

        self.state_embed = nn.Sequential(
            nn.Linear(input_dim * h_window + phase_dim, embed_dim),
            nn.LayerNorm(embed_dim),
            nn.GELU(),
            nn.Linear(embed_dim, embed_dim),
        )

        self.pos_emb = nn.Parameter(torch.randn(1, max_tokens, embed_dim) * 0.02)
        encoder_layer = nn.TransformerEncoderLayer(
            d_model=embed_dim,
            nhead=nhead,
            dim_feedforward=embed_dim * 4,
            dropout=dropout,
            activation='gelu',
            batch_first=True,
            norm_first=True,
        )
        self.transformer = nn.TransformerEncoder(encoder_layer, num_layers=num_layers)

        self.shared_head = nn.Sequential(
            nn.LayerNorm(embed_dim),
            nn.Linear(embed_dim, 160),
            nn.GELU(),
            nn.Dropout(dropout),
        )
        self.arm_head = nn.Sequential(
            nn.Linear(160, 96),
            nn.GELU(),
            nn.Linear(96, 4),
        )
        self.gripper_head = nn.Sequential(
            nn.Linear(160, 64),
            nn.GELU(),
            nn.Linear(64, action_dim - 4),
        )

    def forward(self, demo_seq, demo_mask, cam_hist, progress):
        batch_size, seq_len, _ = demo_seq.shape
        if seq_len + 1 > self.max_tokens:
            raise ValueError(f'Demo sequence is too long: {seq_len}, max is {self.max_tokens - 1}')

        demo_tokens = self.demo_embed(demo_seq) + self.pos_emb[:, 1:seq_len + 1, :]
        state_in = torch.cat([cam_hist, phase_features(progress)], dim=1)
        state_token = self.state_embed(state_in).unsqueeze(1) + self.pos_emb[:, :1, :]

        full_seq = torch.cat([state_token, demo_tokens], dim=1)
        state_mask = torch.zeros(batch_size, 1, dtype=torch.bool, device=demo_mask.device)
        full_mask = torch.cat([state_mask, demo_mask], dim=1)

        out = self.transformer(full_seq, src_key_padding_mask=full_mask)[:, 0, :]
        shared = self.shared_head(out)
        arm_delta = self.arm_head(shared)
        grip_delta = self.gripper_head(shared)
        return torch.cat([arm_delta, grip_delta], dim=1)


dataset = DeltaIntentDataset(raw_data, h_window=H_WINDOW)

all_trials = sorted(set(dataset.trial_numbers))
rng = random.Random(SEED)
shuffled_trials = all_trials[:]
rng.shuffle(shuffled_trials)
val_count = max(1, int(round(0.20 * len(shuffled_trials))))
val_trials = set(shuffled_trials[:val_count])

train_indices = [i for i, trial in enumerate(dataset.trial_numbers) if trial not in val_trials]
val_indices = [i for i, trial in enumerate(dataset.trial_numbers) if trial in val_trials]

print('Validation trials:', sorted(val_trials))
print('Train samples:', len(train_indices), '| Val samples:', len(val_indices))

train_loader = DataLoader(Subset(dataset, train_indices), batch_size=BATCH_SIZE, shuffle=True, num_workers=0)
val_loader = DataLoader(Subset(dataset, val_indices), batch_size=BATCH_SIZE, shuffle=False, num_workers=0)

model = DeltaIntentPolicy(
    input_dim=INPUT_DIM,
    h_window=H_WINDOW,
    phase_dim=PHASE_DIM,
    embed_dim=EMBED_DIM,
    action_dim=ACTION_DIM,
    nhead=NHEAD,
    num_layers=NUM_LAYERS,
    dropout=DROPOUT,
    max_tokens=max(2000, dataset.max_demo_len + 1),
).to(device)

optimizer = optim.AdamW(model.parameters(), lr=LR, weight_decay=WEIGHT_DECAY)
scheduler = optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=EPOCHS)
criterion = nn.SmoothL1Loss(reduction='none', beta=0.5)
loss_weights = torch.tensor([1.0, 1.0, 1.0, 1.0, GRIPPER_LOSS_WEIGHT, GRIPPER_LOSS_WEIGHT], device=device)


def batch_loss(batch, train_mode):
    demo_seq = batch['demo_seq'].to(device)
    demo_mask = batch['demo_mask'].to(device)
    cam_hist = batch['cam_hist'].to(device)
    progress = batch['progress'].to(device)
    target = batch['delta'].to(device)

    if train_mode:
        if CAM_NOISE_STD > 0:
            cam_hist = cam_hist + torch.randn_like(cam_hist) * CAM_NOISE_STD
        if DEMO_NOISE_STD > 0:
            valid = (~demo_mask).unsqueeze(-1).float()
            demo_seq = demo_seq + torch.randn_like(demo_seq) * DEMO_NOISE_STD * valid
        if PROGRESS_NOISE_STD > 0:
            progress = (progress + torch.randn_like(progress) * PROGRESS_NOISE_STD).clamp(0.0, 1.0)

    pred = model(demo_seq, demo_mask, cam_hist, progress)
    weighted = criterion(pred, target) * loss_weights
    return weighted.mean()


best_val = float('inf')
best_state = None

for epoch in range(1, EPOCHS + 1):
    model.train()
    train_loss = 0.0
    for batch in train_loader:
        optimizer.zero_grad(set_to_none=True)
        loss = batch_loss(batch, train_mode=True)
        loss.backward()
        nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
        optimizer.step()
        train_loss += loss.item()

    scheduler.step()
    train_loss /= max(1, len(train_loader))

    model.eval()
    val_loss = 0.0
    with torch.no_grad():
        for batch in val_loader:
            val_loss += batch_loss(batch, train_mode=False).item()
    val_loss /= max(1, len(val_loader))

    if val_loss < best_val:
        best_val = val_loss
        best_state = {k: v.detach().cpu().clone() for k, v in model.state_dict().items()}

    if epoch == 1 or epoch % 25 == 0:
        print(
            f'Epoch {epoch:04d}/{EPOCHS} | '
            f'train {train_loss:.6f} | val {val_loss:.6f} | best {best_val:.6f}'
        )

if best_state is not None:
    model.load_state_dict(best_state)

checkpoint = {
    'model_state': model.state_dict(),
    'delta_mean': torch.tensor(dataset.delta_mean, dtype=torch.float32),
    'delta_std': torch.tensor(dataset.delta_std, dtype=torch.float32),
    'config': {
        'input_dim': INPUT_DIM,
        'action_dim': ACTION_DIM,
        'h_window': H_WINDOW,
        'phase_dim': PHASE_DIM,
        'embed_dim': EMBED_DIM,
        'nhead': NHEAD,
        'num_layers': NUM_LAYERS,
        'dropout': DROPOUT,
        'max_tokens': max(2000, dataset.max_demo_len + 1),
        'model_class': 'DeltaIntentPolicy',
        'feature_mode': 'intent_relative',
        'action_mode': 'delta',
        'separate_heads': True,
        'raw_point_dim': RAW_POINT_DIM,
        'intent_feature_dim': INTENT_FEATURE_DIM,
    },
    'training': {
        'epochs': EPOCHS,
        'batch_size': BATCH_SIZE,
        'lr': LR,
        'weight_decay': WEIGHT_DECAY,
        'best_val_loss': best_val,
        'samples': len(dataset),
        'trials': sorted(set(dataset.trial_numbers)),
        'validation_trials': sorted(val_trials),
    },
}

torch.save(checkpoint, MODEL_OUT)
print(f'Saved: {MODEL_OUT.resolve()}')

if IN_COLAB:
    files.download(str(MODEL_OUT))
